Gradio app to review images generated by stable diffusion

This was generated using Claude Sonnet 4.5 on Typing Mind.

It is not yet validated for function, but it does create a working Gradio interface.

In [ ]:
import gradio as gr
import sqlite3
import json
import os
from pathlib import Path
from datetime import datetime
import hashlib
from PIL import Image
import pandas as pd

In [ ]:
class ImageRatingDB:
    def __init__(self, db_path="image_ratings.db"):
        self.db_path = db_path
        self.init_db()
    
    def init_db(self):
        """Initialize the database with required tables"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS image_ratings (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                unique_id TEXT UNIQUE NOT NULL,
                image_path TEXT NOT NULL,
                
                -- Rating criteria (0=N/A, 1-5 scale)
                aesthetic_quality INTEGER DEFAULT 0,
                prompt_adherence INTEGER DEFAULT 0,
                technical_quality INTEGER DEFAULT 0,
                style_consistency INTEGER DEFAULT 0,
                
                -- Prompts
                positive_prompt TEXT,
                negative_prompt TEXT,
                
                -- Generation parameters
                checkpoint_model TEXT,
                loras TEXT,
                cfg_scale REAL,
                seed INTEGER,
                height INTEGER,
                width INTEGER,
                clip_skip INTEGER,
                sampler TEXT,
                scheduler TEXT,
                
                -- Metadata
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        
        conn.commit()
        conn.close()


    def generate_unique_id(self, json_data):
        """Generate a unique identifier from the generation parameters"""
        # Create a hash from key generation parameters
        key_params = {
            'seed': json_data.get('seed', ''),
            'prompt': json_data.get('prompt', ''),
            'negative_prompt': json_data.get('negative_prompt', ''),
            'cfg_scale': json_data.get('cfg_scale', ''),
            'steps': json_data.get('steps', ''),
            'sampler': json_data.get('sampler_name', ''),
            'model': json_data.get('sd_model_name', '')
        }
        
        hash_string = json.dumps(key_params, sort_keys=True)
        return hashlib.sha256(hash_string.encode()).hexdigest()

    def get_rating(self, unique_id):
        """Retrieve existing rating for an image"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute('''
            SELECT aesthetic_quality, prompt_adherence, technical_quality, 
                   style_consistency
            FROM image_ratings 
            WHERE unique_id = ?
        ''', (unique_id,))
        
        result = cursor.fetchone()
        conn.close()
        
        if result:
            return {
                'aesthetic_quality': result[0],
                'prompt_adherence': result[1],
                'technical_quality': result[2],
                'style_consistency': result[3]
            }
        return None


    def save_rating(self, unique_id, image_path, ratings, json_data):
        """Save or update a rating"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Extract LORA information if present
        loras_str = json.dumps(json_data.get('loras', []))
        
        cursor.execute('''
            INSERT INTO image_ratings (
                unique_id, image_path,
                aesthetic_quality, prompt_adherence, technical_quality, style_consistency,
                positive_prompt, negative_prompt,
                checkpoint_model, loras, cfg_scale, seed, height, width,
                clip_skip, sampler, scheduler
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(unique_id) DO UPDATE SET
                image_path = excluded.image_path,
                aesthetic_quality = excluded.aesthetic_quality,
                prompt_adherence = excluded.prompt_adherence,
                technical_quality = excluded.technical_quality,
                style_consistency = excluded.style_consistency,
                updated_at = CURRENT_TIMESTAMP
        ''', (
            unique_id, image_path,
            ratings['aesthetic_quality'], ratings['prompt_adherence'],
            ratings['technical_quality'], ratings['style_consistency'],
            json_data.get('prompt', ''),
            json_data.get('negative_prompt', ''),
            json_data.get('sd_model_name', ''),
            loras_str,
            json_data.get('cfg_scale', 0),
            json_data.get('seed', 0),
            json_data.get('height', 0),
            json_data.get('width', 0),
            json_data.get('clip_skip', 0),
            json_data.get('sampler_name', ''),
            json_data.get('scheduler', '')
        ))
        
        conn.commit()
        conn.close()


    def get_all_ratings(self):
        """Get all ratings for analysis"""
        conn = sqlite3.connect(self.db_path)
        df = pd.read_sql_query("SELECT * FROM image_ratings", conn)
        conn.close()
        return df

    

In [ ]:
#Initialize database
db = ImageRatingDB()

In [ ]:
# Cell 5: Rating interface functions
def load_image_data(json_file):
    """Load image and metadata from JSON response file"""
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        # Extract the image path (assumes image is in same directory as JSON)
        json_path = Path(json_file)
        
        # Look for PNG file with same name
        image_path = json_path.with_suffix('.png')
        if not image_path.exists():
            # Try to find any image file in the same directory with similar name
            possible_images = list(json_path.parent.glob(f"{json_path.stem}.*"))
            image_files = [f for f in possible_images if f.suffix.lower() in ['.png', '.jpg', '.jpeg']]
            if image_files:
                image_path = image_files[0]
            else:
                return None, "Image file not found", {}, None, None
        
        # Load image
        image = Image.open(image_path)
        
        # Extract info from JSON (Auto1111 format)
        info = data.get('info', {})
        if isinstance(info, str):
            info = json.loads(info)
        
        # Generate unique ID
        unique_id = db.generate_unique_id(info)
        
        # Check for existing rating
        existing_rating = db.get_rating(unique_id)
        
        # Format metadata display
        metadata = f"""
**Generation Parameters:**
- **Model:** {info.get('sd_model_name', 'N/A')}
- **Sampler:** {info.get('sampler_name', 'N/A')}
- **Scheduler:** {info.get('scheduler', 'N/A')}
- **CFG Scale:** {info.get('cfg_scale', 'N/A')}
- **Seed:** {info.get('seed', 'N/A')}
- **Size:** {info.get('width', 'N/A')} x {info.get('height', 'N/A')}
- **Clip Skip:** {info.get('clip_skip', 'N/A')}

**Positive Prompt:**
{info.get('prompt', 'N/A')}

**Negative Prompt:**
{info.get('negative_prompt', 'N/A')}
"""
        
        # Add LORA information if available
        loras = info.get('loras', [])
        if loras:
            metadata += f"\n**LORAs:** {', '.join([l.get('name', 'Unknown') for l in loras])}"
        
        return image, metadata, info, str(image_path), unique_id, existing_rating
        
    except Exception as e:
        return None, f"Error loading data: {str(e)}", {}, None, None, None

In [ ]:
# Global variables to store current session data
current_image_path = None
current_info = None
current_unique_id = None


In [ ]:
def process_json_upload(json_file):
    """Process uploaded JSON file and display image with metadata"""
    global current_image_path, current_info, current_unique_id
    
    if json_file is None:
        return None, "Please upload a JSON file", 0, 0, 0, 0, "Save"
    
    image, metadata, info, img_path, unique_id, existing_rating = load_image_data(json_file.name)
    
    current_image_path = img_path
    current_info = info
    current_unique_id = unique_id
    
    if existing_rating:
        # Load existing ratings
        return (
            image, 
            metadata,
            existing_rating['aesthetic_quality'],
            existing_rating['prompt_adherence'],
            existing_rating['technical_quality'],
            existing_rating['style_consistency'],
            "Update"
        )
    else:
        # Default values
        return image, metadata, 0, 0, 0, 0, "Save"

In [ ]:
def save_ratings(aesthetic, prompt_adh, technical, style_cons):
    """Save ratings to database"""
    global current_image_path, current_info, current_unique_id
    
    if current_unique_id is None:
        return "Error: No image loaded. Please upload a JSON file first."
    
    ratings = {
        'aesthetic_quality': aesthetic,
        'prompt_adherence': prompt_adh,
        'technical_quality': technical,
        'style_consistency': style_cons
    }
    
    try:
        db.save_rating(current_unique_id, current_image_path, ratings, current_info)
        return "✅ Ratings saved successfully!"
    except Exception as e:
        return f"❌ Error saving ratings: {str(e)}"

In [ ]:
def cancel_ratings():
    """Cancel and reset the form"""
    global current_image_path, current_info, current_unique_id
    current_image_path = None
    current_info = None
    current_unique_id = None
    return None, "", 0, 0, 0, 0, "Ratings cleared. Upload a new JSON file."

In [ ]:
def create_rating_interface():
    with gr.Blocks(title="SD Image Rating System", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🎨 Stable Diffusion Image Rating System")
        gr.Markdown("Upload the JSON response file from Auto1111 to rate generated images.")
        
        with gr.Row():
            with gr.Column(scale=1):
                json_input = gr.File(
                    label="Upload Auto1111 JSON Response",
                    file_types=[".json"]
                )
                
                image_display = gr.Image(
                    label="Generated Image",
                    type="pil",
                    height=400
                )
                
            with gr.Column(scale=1):
                metadata_display = gr.Markdown("Upload a JSON file to see image details")
                
        gr.Markdown("## 📊 Rate the Image")
        gr.Markdown("Rate each criterion from 1 (horrible) to 5 (great), or select N/A (0)")
        
        with gr.Row():
            with gr.Column():
                aesthetic_rating = gr.Radio(
                    choices=[
                        ("N/A", 0),
                        ("1 - Horrible", 1),
                        ("2 - Bad", 2),
                        ("3 - Neutral", 3),
                        ("4 - Good", 4),
                        ("5 - Great", 5)
                    ],
                    label="🎭 Aesthetic Quality / Beauty",
                    value=0
                )
                
                prompt_rating = gr.Radio(
                    choices=[
                        ("N/A", 0),
                        ("1 - Horrible", 1),
                        ("2 - Bad", 2),
                        ("3 - Neutral", 3),
                        ("4 - Good", 4),
                        ("5 - Great", 5)
                    ],
                    label="🎯 Prompt Adherence",
                    value=0
                )
                
            with gr.Column():
                technical_rating = gr.Radio(
                    choices=[
                        ("N/A", 0),
                        ("1 - Horrible", 1),
                        ("2 - Bad", 2),
                        ("3 - Neutral", 3),
                        ("4 - Good", 4),
                        ("5 - Great", 5)
                    ],
                    label="🔧 Technical Quality (No Defects)",
                    value=0
                )
                
                style_rating = gr.Radio(
                    choices=[
                        ("N/A", 0),
                        ("1 - Horrible", 1),
                        ("2 - Bad", 2),
                        ("3 - Neutral", 3),
                        ("4 - Good", 4),
                        ("5 - Great", 5)
                    ],
                    label="🎨 Style Consistency",
                    value=0
                )
        
        status_message = gr.Textbox(label="Status", interactive=False)
        
        with gr.Row():
            save_btn = gr.Button("Save", variant="primary", size="lg")
            cancel_btn = gr.Button("Cancel", variant="secondary", size="lg")
        
        # Event handlers
        json_input.change(
            fn=process_json_upload,
            inputs=[json_input],
            outputs=[
                image_display,
                metadata_display,
                aesthetic_rating,
                prompt_rating,
                technical_rating,
                style_rating,
                save_btn
            ]
        )
        
        save_btn.click(
            fn=save_ratings,
            inputs=[aesthetic_rating, prompt_rating, technical_rating, style_rating],
            outputs=[status_message]
        )
        
        cancel_btn.click(
            fn=cancel_ratings,
            inputs=[],
            outputs=[
                image_display,
                metadata_display,
                aesthetic_rating,
                prompt_rating,
                technical_rating,
                style_rating,
                status_message
            ]
        )
    
    return demo


In [ ]:
rating_app = create_rating_interface()
rating_app.launch(share=False, inbrowser=True)